In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)


In [3]:
def kullback_leibler_conj(gamma,s,t,constraints):
    N = s.shape[0]
    w = cp.Variable(N)
    constraints.append(w - gamma*(np.zeros(N)+1) <= t)
    for i in range(N):
        constraints.append(cp.kl_div(gamma,w[i])+gamma+s[i]-w[i]<= 0)
    return(constraints)

def kullback_leibler(p_i,q_i,par,phi_cons):
    phi_cons = phi_cons -cp.entr(q_i) - q_i*np.log(p_i)
    return(phi_cons)

In [4]:
def h_sing_power(z1,z2,par,constraints):
    v = 1-cp.power((1-cp.sum(z2)),par)
    constraints.append(cp.sum(z1)-v <= 0)
    return(constraints)

def h_sing_power_conj(lbda,v,z,par,constraints):
    M = lbda.shape[0]
    xi_2 = cp.Variable(M, nonneg = True)
    xi_3 = cp.Variable(M, nonneg = True)
    xi_4 = cp.Variable(M, nonneg = True)
    constraints.extend((xi_2 >= xi_3, xi_3 <= v))
    constraints.append(lbda-xi_3+(par**(-1/(par-1))-par**(-par/(par-1)))*xi_4 <= z)
    exponent = np.array([(par-1)/par,1-(par-1)/par])
    for j in range(M):
        constraints.append(xi_2[j]-cp.geo_mean(cp.vstack([xi_4[j],lbda[j]]),exponent)<= 0)
    return(constraints)

def h_quad_conj(lbda,v,z,par,constraints):
    M = lbda.shape[0]
    eta = cp.Variable(M, nonneg= True)
    for j in range(M):
        constraints.append(cp.norm(cp.vstack([eta[j],(z[j]-lbda[j])/2]))<=(z[j]+lbda[j])/2)
        constraints.append(1/(2*np.sqrt(m))*(-v[j]+lbda[j]+par*lbda[j])<= eta[j])
    return(constraints)

def h_quadratic(z1,z2,par,constraints):
    v = (1+par)*cp.sum(z2)-par*cp.sum(z2)**2
    constraints.append(cp.sum(z1)-v <= 0)
    return(constraints)

In [5]:
def robustcheck(a,R,r,p,par,r_f,h_func, phi_func):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort((-x))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        constraints = h_func(z1,z2,par,constraints)
        phi_cons = phi_func(p[i],q[i],par,phi_cons)
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(-q_b.T @ (R@a + (1-cp.sum(a))*r_f))
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value)

In [6]:
def dual(sets,a,p,R,r,par,r_f,c,phi_conj, h_conj):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable(M)
    lbda = cp.Variable(M, nonneg = True)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N)
    z = cp.Variable(M)
    s = cp.Variable(N)
    constraints = []
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        constraints.append(-(R.dot(a) + (1-sum(a))*r_f)[i] - lbdasum - beta <= 0)
        constraints.append(s[i] == -alpha + vsum)
    constraints = phi_conj(gamma,s,t,constraints)
    constraints = h_conj(lbda,v,z,par,constraints)
    obj = cp.Minimize(alpha + beta + gamma * r  + cp.sum(z) + p@t)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value)

In [7]:
np.random.seed(10)

In [8]:
N=6
p = (np.zeros(N)+1)*1/N
I = 2
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))


[0.08701171 0.08745984]


In [9]:
w = np.zeros(I)+1/I
r = 0.043
par = 4
r_f = 0.001
c = 0.3
h_func = h_sing_power
phi_func = kullback_leibler
phi_conj = kullback_leibler_conj
h_conj = h_sing_power_conj

In [10]:
robustcheck(w,R,r,p,par,r_f,h_func, phi_func)

0.06604832333868521

In [11]:
oldrank = np.argsort(R.dot(w))
sets = ranktoset(oldrank)

In [12]:
dual(sets,w,p,R,r,par,r_f,c,phi_conj, h_conj)

0.06604832939396278

In [14]:
x=np.arange(1,N)
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]

In [21]:
sets = psets
dual (sets,p,R,r,m,r_f,c)

(array([2.59466842, 7.89932368]), 0.9071461509844918)

In [54]:
[probv,vv,lbdav,alphav,betav,gammav,tv]=dual (sets,p,R,r,m,r_f,a)
N = len(p)
M = len(sets)
cons1 =np.zeros(N)
cons2 = np.zeros(N)
z0 = 0
for j in range(M):
    z9 = -np.min(vv[j,sets[j]])*(1-m)+lbdav[j]
    z0 = z0 + max(z9,0)
for i in range(N):
    lbdsom = 0
    for j in range(M):
        if i in sets[j]:
            lbdsom = lbdsom + lbdav[j]
    cons1[i] = R.dot(a)[i] + betav + lbdsom
    cons2[i] = gammav * np.exp((-alphav+sum(vv[0:M:1,i]))/gammav)-tv[i]
print(cons1)
print(cons2)
print(-1+alphav+betav+gammav*r+sum(p*tv)+z0)

[ 5.70003067e-09 -5.38403810e-09  6.53539289e+00  1.70622106e+01]
[-8.10329546e-08 -2.98001260e-06 -3.94966149e-08 -2.06583066e-08]
[24.65265508]
